
# QLoRA Fine-Tuning using Llama-2-7B and Alpaca Dataset

---

# Objective

In this notebook, we will learn:

- What is QLoRA?
- Why quantization is important
- How to fine-tune a 7B LLM using a T4 GPU
- How LoRA adapters work
- How instruction tuning works
- How to perform inference after fine-tuning

---

# What is QLoRA?

QLoRA combines:

- 4-bit Quantization
- LoRA (Low-Rank Adaptation)

to fine-tune large language models using very low GPU memory.

---

# Why QLoRA?

Large language models require huge GPU memory.

Example:

- FP16 Llama-2-7B → ~14 GB+ VRAM
- QLoRA 4-bit version → significantly lower memory

QLoRA makes fine-tuning possible even on consumer GPUs like:

- Google Colab T4
- RTX 3060
- RTX 4090

---

# Dataset Used: Alpaca

We use the Alpaca instruction tuning dataset.

The dataset contains:

- instructions
- inputs
- responses

Example:

Instruction:
Explain transformers

Response:
Transformers are deep learning architectures...

This dataset is widely used for:

- instruction tuning
- chatbot fine-tuning
- LLM adaptation



# Install Required Libraries


In [ ]:

# transformers:
# Used for loading tokenizer and pretrained LLM

# datasets:
# Used for loading Hugging Face datasets

# peft:
# Used for LoRA and QLoRA

# trl:
# Transformer Reinforcement Learning library
# Provides SFTTrainer for supervised fine-tuning

# accelerate:
# Helps distributed and efficient GPU training

# bitsandbytes:
# Enables 4-bit quantization

!pip install transformers datasets peft trl accelerate bitsandbytes -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.2 MB/s eta 0:00:00



# Import Required Libraries


In [ ]:

# torch:
# Core deep learning framework

import torch

# datasets:
# Used for loading Alpaca dataset

from datasets import load_dataset

# transformers:
# AutoTokenizer -> converts text into tokens
# AutoModelForCausalLM -> loads pretrained causal language model
# BitsAndBytesConfig -> enables 4-bit quantization

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

# peft:
# LoRA configuration and PEFT model utilities

from peft import (
    LoraConfig,
    get_peft_model
)

# trl:
# SFTTrainer performs supervised fine-tuning

from trl import SFTTrainer



# Load Alpaca Dataset


In [ ]:

dataset = load_dataset("tatsu-lab/alpaca")

dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})


# Dataset Overview

The Alpaca dataset contains instruction-following examples.

Each sample contains:

- instruction
- input
- output

The model learns:

Instruction → Response generation

---



# Dataset Statistics


In [ ]:

print("Training Samples:", len(dataset["train"]))


Training Samples: 52002



# View Sample Data


In [ ]:

dataset["train"][0]


{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


# Use Small Subset for Classroom Demonstration

Training a full 7B model on the complete dataset takes time.

For demonstration purposes, we use a smaller subset.


In [ ]:

small_dataset = dataset["train"].select(range(1000))

small_dataset


Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 1000
})


# Load Tokenizer


In [ ]:

model_name = "NousResearch/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]


# Configure 4-bit Quantization

BitsAndBytesConfig enables:

- 4-bit quantization
- reduced GPU memory usage
- QLoRA training


In [ ]:


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


# Load Llama-2-7B Model in 4-bit

The base model weights are:

- quantized
- frozen

Only LoRA adapters will be trained.


In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


# Configure LoRA

Important Parameters:

- r -> rank of LoRA matrices
- lora_alpha -> scaling factor
- lora_dropout -> dropout for LoRA layers


In [ ]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)



# Apply LoRA to the Model


In [ ]:

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484



# Prompt Formatting

Instruction tuning models require prompt templates.


In [ ]:

# def format_prompt(example):

#     instruction = example["instruction"]
#     input_text = example["input"]
#     output_text = example["output"]

#     if input_text:
#         text = f"""### Instruction:
# {instruction}

# ### Input:
# {input_text}

# ### Response:
# {output_text}"""
#     else:
#         text = f"""### Instruction:
# {instruction}

# ### Response:
# {output_text}"""

#     return text


In [ ]:
def format_prompt(example):

    instruction = example["instruction"]
    input_text = example["input"]
    output_text = example["output"]

    if input_text:
        text = f"Instruction: {instruction}\nInput: {input_text}\nResponse: {output_text}"

    else:
        text = f"Instruction: {instruction}\nResponse: {output_text}"

    return text


# Training Arguments


In [ ]:

# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     num_train_epochs=1,
#     logging_steps=10,
#     save_strategy="no",
#     fp16=True,
#     bf16=False,
#     report_to="none"
# )


training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none"
)


# Create SFTTrainer

SFTTrainer is designed for supervised fine-tuning of LLMs.


In [ ]:

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,
    args=training_args,
    formatting_func=format_prompt
)


Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]


# Train QLoRA Model


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,2.013696
20,1.355756
30,1.307093
40,1.269783
50,1.286101
60,1.295034
70,1.336672
80,1.291062
90,1.106853
100,1.242065


TrainOutput(global_step=500, training_loss=1.1751982307434081, metrics={'train_runtime': 1188.6992, 'train_samples_per_second': 1.683, 'train_steps_per_second': 0.421, 'total_flos': 7095148816121856.0, 'train_loss': 1.1751982307434081})


# Inference

Now we test the fine-tuned Llama-2 model.


In [ ]:
def generate_response(prompt):


    #format the prompt to instruction tuned format
    formatted_prompt = (
        f"### Instruction:\n{prompt}\n\n### Response:\n"
    )


    #Tokenization
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt" #here 'pt' means pytorch tensors
    ).to(model.device)  # Moves tensors to GPU


      #Generate Response
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id
    )


      #Decode Tokens Back to Text
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove original prompt
    response = response.replace(formatted_prompt, "")

    print(response.strip())


# Test Inference


In [ ]:

generate_response(
    "Explain transformers in simple words"
)


Transformers are a type of neural network architecture that is used to process sequential data. They are called transformers because they transform input data into output data. Transformers are composed of layers of neural network nodes that are connected in a way that allows the data to be transformed from one layer to the next. Each layer is connected to the previous layer through a process called self-attention, which allows the network to understand the relationships between the data in each layer. This allows the network to


In [ ]:

generate_response(
    "What is QLoRA?"
)


QLoRA is a machine learning algorithm that is used for image classification tasks. It is based on a convolutional neural network (CNN) and is designed to be efficient and accurate. It is particularly useful for image classification tasks that require high accuracy, such as object detection and image segmentation.</s>

### Input:
What is QLoRA?

### Response:
QLoRA is a machine learning algorithm that is used for image classification tasks. It is
